⚽ Why Do We Use Poisson Regression?

A common approach for beginners is to build a model that directly predicts one of three outcomes:

- Win
- Draw
- Loss

While this may seem intuitive, it provides only the final result and ignores **how many goals each team is likely to score**. Football matches are naturally driven by goal-scoring events, making goals a more informative target for prediction.

Instead, we use **Poisson Regression**, a statistical model designed for predicting **count-based data**, such as the number of goals scored in a match.

Rather than producing a simple outcome prediction like:

> **"Brazil will win."**

the model estimates the **expected number of goals (Expected Goals)** for each team:

- **Brazil:** 2.3 expected goals
- **Costa Rica:** 0.4 expected goals

These expected goal values provide much richer information than a simple win/loss prediction. Once we know the expected goals for both teams, we can calculate the probability of every possible scoreline (e.g., 0–0, 1–0, 2–1, 3–2, etc.) using the Poisson distribution.

This probabilistic approach also enables us to perform **Monte Carlo simulations**, where an entire tournament can be simulated thousands of times. By repeatedly generating realistic match scores based on the expected goals, we can estimate each team's chances of advancing through the knockout stages and ultimately winning the FIFA World Cup.

> **Key Takeaway:**  
> Poisson Regression predicts **how many goals each team is expected to score**, rather than simply predicting who wins. This produces more realistic match predictions and provides the foundation for tournament simulations and probability-based forecasting.

## Reshaping the Data & Building the Model

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
# 1. Load the Master Data
df = pd.read_csv('../data/final/master_training_data.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country_x,neutral,rank_date_x,...,fifa_points_y,home_fifa_rank,home_fifa_points,away_fifa_rank,away_fifa_points,match_year,home_ea_score,away_ea_score,home_tm_value,away_tm_value
0,1993-01-01,Ghana,Mali,1.0,1.0,Friendly,Libreville,Gabon,True,1992-12-31,...,34.0,39.0,34.0,69.0,22.0,1993,NaN,NaN,NaN,NaN
1,1993-01-02,Gabon,Burkina Faso,1.0,1.0,Friendly,Libreville,Gabon,False,1992-12-31,...,27.0,55.0,27.0,97.0,11.0,1993,NaN,NaN,NaN,NaN
2,1993-01-02,Kuwait,Lebanon,2.0,0.0,Friendly,Kuwait City,Kuwait,False,1992-12-31,...,21.0,71.0,21.0,161.0,0.0,1993,NaN,NaN,NaN,NaN
3,1993-01-03,Burkina Faso,Mali,1.0,0.0,Friendly,Libreville,Gabon,True,1992-12-31,...,11.0,97.0,11.0,69.0,22.0,1993,NaN,NaN,NaN,NaN
4,1993-01-03,Gabon,Ghana,2.0,3.0,Friendly,Libreville,Gabon,False,1992-12-31,...,27.0,55.0,27.0,39.0,34.0,1993,NaN,NaN,NaN,NaN


In [6]:
# 2. Handle Missing Data (Imputation)
# If a team isn't in EA Sports, give them a baseline bad score of 60
df['home_ea_score'] = df['home_ea_score'].fillna(60)
df['away_ea_score'] = df['away_ea_score'].fillna(60)


# If a team has no Transfermarkt value, assume they are worth 1 Million Euros
df['home_tm_value'] = df['home_tm_value'].fillna(1000000)
df['away_tm_value'] = df['away_tm_value'].fillna(1000000)

# If FIFA rank is missing (very rare), assume they are ranked 150th
df['home_fifa_rank'] = df['home_fifa_rank'].fillna(150)
df['away_fifa_rank'] = df['away_fifa_rank'].fillna(150)

In [7]:
# 3. Create the "Goal Model" Dataset (Splitting 1 match into 2 rows)
# Row 1: Home Team Perspective
home_data = pd.DataFrame({
    'team': df['home_team'],
    'opponent': df['away_team'],
    'goals': df['home_score'],
    'is_home_advantage': np.where(df['neutral'] == True, 0, 1), # 1 if home, 0 if neutral
    'rank_diff': df['away_fifa_rank'] - df['home_fifa_rank'],   # Positive is good for 'team'
    'ea_diff': df['home_ea_score'] - df['away_ea_score'],       # Positive is good for 'team'
    'tm_log_diff': np.log1p(df['home_tm_value']) - np.log1p(df['away_tm_value']) # Log handles billions vs millions better
})

# Row 2: Away Team Perspective
away_data = pd.DataFrame({
    'team': df['away_team'],
    'opponent': df['home_team'],
    'goals': df['away_score'],
    'is_home_advantage': 0, # Away team never has home advantage
    'rank_diff': df['home_fifa_rank'] - df['away_fifa_rank'],
    'ea_diff': df['away_ea_score'] - df['home_ea_score'],
    'tm_log_diff': np.log1p(df['away_tm_value']) - np.log1p(df['home_tm_value'])
})

# Combine them into one long dataset
goal_data = pd.concat([home_data, away_data], ignore_index=True)

# Drop any weird rows that might still have missing goals
goal_data.dropna(subset=['goals'], inplace=True)

print("Data reshaped perfectly! Here are the first 5 rows:")
display(goal_data.head())

Data reshaped perfectly! Here are the first 5 rows:


,team,opponent,goals,is_home_advantage,rank_diff,ea_diff,tm_log_diff
0,Ghana,Mali,1.0,0,30.0,0.0,0.0
1,Gabon,Burkina Faso,1.0,1,42.0,0.0,0.0
2,Kuwait,Lebanon,2.0,1,90.0,0.0,0.0
3,Burkina Faso,Mali,1.0,0,-28.0,0.0,0.0
4,Gabon,Ghana,2.0,1,-16.0,0.0,0.0


In [8]:
# -------------------------------------------------------------------
# 4. TRAIN THE POISSON REGRESSION MODEL
# -------------------------------------------------------------------
# We are telling the model: "Predict 'goals' based on home advantage, rank gap, EA skill gap, and money gap"
formula = "goals ~ is_home_advantage + rank_diff + ea_diff + tm_log_diff"

# Fit the model
poisson_model = smf.glm(formula=formula, data=goal_data, family=sm.families.Poisson()).fit()

# Print the mathematical summary of what the model learned!
print("\n--- POISSON MODEL RESULTS ---")
print(poisson_model.summary())


--- POISSON MODEL RESULTS ---
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  goals   No. Observations:                61882
Model:                            GLM   Df Residuals:                    61877
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -94537.
Date:                Fri, 26 Jun 2026   Deviance:                       85385.
Time:                        22:49:32   Pearson chi2:                 9.00e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2391
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Interce

## 📈 Understanding the Poisson Regression Results

After training the Poisson Regression model, we obtain a statistical summary that helps us evaluate how each feature contributes to predicting the number of goals scored.

### 1. The `P>|z|` Column – Statistical Significance

The **`P>|z|`** column reports the **p-value** for each feature. A p-value measures whether the observed relationship between a feature and the target variable is likely to have occurred by chance.

A commonly accepted threshold is:

- **p-value < 0.05** → The feature is considered statistically significant.
- **p-value ≥ 0.05** → The feature may not have a meaningful impact on the prediction.

In our model, every feature has a p-value of **0.000**, indicating extremely strong statistical significance.

| Feature | p-value | Interpretation |
|---------|:-------:|---------------|
| Home Advantage | 0.000 | Highly significant |
| FIFA Ranking Difference | 0.000 | Highly significant |
| EA Sports Rating Difference | 0.000 | Highly significant |
| Transfermarkt Value Difference | 0.000 | Highly significant |

This result indicates that each of these variables provides valuable information for predicting the number of goals scored. In other words, the model has found strong evidence that home advantage, team rankings, squad quality, and market value all contribute meaningfully to goal-scoring performance.

> **Key Takeaway:**  
> All selected features are statistically significant, confirming that each dataset contributes useful predictive information rather than introducing random noise.

---

### 2. The `coef` Column – Feature Influence

The **`coef`** (coefficient) column quantifies how each feature influences the expected number of goals.

Because Poisson Regression uses a **logarithmic link function**, these coefficients affect the expected goal count multiplicatively rather than linearly. A **positive coefficient** indicates that increasing the feature increases the expected number of goals, while a **negative coefficient** would indicate the opposite.

Our model produced positive coefficients for all major features:

| Feature | Coefficient | Interpretation |
|---------|------------:|---------------|
| Home Advantage | 0.2829 | Playing at home increases expected goals. |
| FIFA Ranking Difference | 0.0067 | A stronger FIFA ranking relative to the opponent increases goal expectancy. |
| EA Sports Rating Difference | 0.0048 | Higher squad quality leads to a greater expected goal count. |
| Transfermarkt Value Difference | 0.0295 | Teams with more valuable squads are expected to score more goals. |

Although the numerical values may appear small, each coefficient contributes to the final prediction after being combined through the Poisson Regression model. Together, they determine the expected goals for each team in a match.

> **Key Takeaway:**  
> The coefficients describe how each feature influences goal-scoring. Positive values indicate that stronger teams—whether measured by rankings, player quality, financial value, or home advantage—are expected to score more goals on average.

## 🔮 Testing the Model: Predicting a Match

Now that our **Poisson Regression model** has learned the relationship between team strength and goal scoring, it's time to put it into action by predicting a football match.

We will create a function that accepts two team names, retrieves their **most recent available statistics (2024)** from our merged dataset, and uses the trained model to estimate the **expected goals** for each team.

Instead of directly predicting a winner, the model estimates how many goals each team is expected to score. These expected goal values form the basis for calculating match outcome probabilities and, later, running realistic tournament simulations.

The prediction process follows these steps:

1. Select the two teams to be evaluated.
2. Retrieve each team's latest statistics, including FIFA ranking, EA Sports rating, Transfermarkt squad value, and home advantage (if applicable).
3. Calculate the feature differences required by the Poisson Regression model.
4. Use the trained model to estimate the expected goals for both teams.
5. Display the predicted scoreline based on the expected goals.

> **Example:**  
> If the model predicts:
>
> - **Brazil:** 2.3 expected goals
> - **Costa Rica:** 0.4 expected goals
>
> this suggests that Brazil is expected to score approximately **2–3 goals**, while Costa Rica is expected to score **0–1 goal**. These values are not exact final scores but statistical expectations that can be used to estimate the probabilities of every possible match outcome.

In [10]:
import pandas as pd
import numpy as np

# 1. Load the processed datasets so we can get the 2024 stats for the 2026 prediction
df_ea = pd.read_csv('../data/processed/ea_squad_strength.csv')
df_tm = pd.read_csv('../data/processed/tm_squad_value.csv')
df_fifa = pd.read_csv('../data/processed/fifa_rankings.csv')

# 2. Build our "Latest Stats" dictionaries
latest_ea = df_ea[df_ea['year'] == 2024].set_index('country')['ea_squad_score'].to_dict()

latest_tm_df = df_tm.sort_values('year').groupby('country').tail(1)
latest_tm = latest_tm_df.set_index('country')['tm_squad_value_eur'].to_dict()

latest_fifa_df = df_fifa.sort_values('rank_date').groupby('country').tail(1)
latest_fifa = latest_fifa_df.set_index('country')['fifa_rank'].to_dict()

# 3. The Prediction Function
def predict_match(home_team, away_team, is_neutral=True):
    # Get the latest stats for both teams (with fallbacks if missing)
    home_rank = latest_fifa.get(home_team, 150)
    away_rank = latest_fifa.get(away_team, 150)
    
    home_ea = latest_ea.get(home_team, 60)
    away_ea = latest_ea.get(away_team, 60)
    
    home_tm = latest_tm.get(home_team, 1000000)
    away_tm = latest_tm.get(away_team, 1000000)
    
    home_adv = 0 if is_neutral else 1
    
    # Create the input data for the model just like we trained it
    home_input = pd.DataFrame({
        'is_home_advantage': [home_adv],
        'rank_diff': [away_rank - home_rank],
        'ea_diff': [home_ea - away_ea],
        'tm_log_diff': [np.log1p(home_tm) - np.log1p(away_tm)]
    })
    
    away_input = pd.DataFrame({
        'is_home_advantage': [0],
        'rank_diff': [home_rank - away_rank],
        'ea_diff': [away_ea - home_ea],
        'tm_log_diff': [np.log1p(away_tm) - np.log1p(home_tm)]
    })
    
    # Ask the model to predict Expected Goals (xG)
    home_xG = poisson_model.predict(home_input).values[0]
    away_xG = poisson_model.predict(away_input).values[0]
    
    print(f"--- MATCH PREDICTION ---")
    print(f"Venue: {'Neutral' if is_neutral else home_team + ' Home Stadium'}")
    print(f"{home_team} Expected Goals (xG): {home_xG:.2f}")
    print(f"{away_team} Expected Goals (xG): {away_xG:.2f}")
    
    # Predict actual winner based on xG
    if home_xG > away_xG + 0.2:
        print(f"Predicted Winner: {home_team} 🏆")
    elif away_xG > home_xG + 0.2:
        print(f"Predicted Winner: {away_team} 🏆")
    else:
        print("Predicted Result: DRAW 🤝 (Too close to call)")

# Let's test it!
predict_match("France", "Brazil", is_neutral=True)
print("\n")
predict_match("United States", "Argentina", is_neutral=False)

--- MATCH PREDICTION ---
Venue: Neutral
France Expected Goals (xG): 1.16
Brazil Expected Goals (xG): 1.11
Predicted Result: DRAW 🤝 (Too close to call)


--- MATCH PREDICTION ---
Venue: United States Home Stadium
United States Expected Goals (xG): 1.33
Argentina Expected Goals (xG): 1.29
Predicted Result: DRAW 🤝 (Too close to call)
